<a href="https://colab.research.google.com/github/Sundeep-Parmar-UC/AgeCare-ACIL/blob/main/ACIL_Excel_Prototype22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, we need to install the `openpyxl` library, which pandas uses to read and write Excel files.

In [98]:
import pandas as pd
import re # Import the regular expression module
import colorsys
import numpy as np
try:
   import openpyxl
except:
  !pip install openpyxl

try:
   import xlsxwriter
except:
  !pip install xlsxwriter




In [99]:
from functions import *

In [100]:
#Global Variable Definitions

#Loaded worksheets: A dictionary where keys are source filenames and values are dictionaries of DataFrames (each representing a sheet from that file).
all_worksheets_df = {} # Re-initialize to ensure it's empty before populating for multiple files

#Source File Name:
SourceFile = []
SourceFile.append("ACIL-source.xlsx")
SourceFilePath = "/content/"

#Destination File Name:
DestinationFile = "Carlos-Summary.xlsx"
DestinationFilePath = "/content/"

#Monthly File Name:
NextMonthTemplateFile = "MonthTemplate.xlsx"
NextMonthTemplateFilePath = "/content/"


new_excel_file_path = DestinationFilePath + DestinationFile

#set Month to summarize
MonthList = {}
RevisedMonthList = {}

#LedgerColumn,  the column that contains the financial categories
LedgerColumn = -1

#MonthColumn,  the column that contains the finanacial actual data for the month
MonthColumn = -1

#SiteList
SiteList = {}
#SiteList = ["BBC", "BJP", "BMT", "BMC", "BMR", "BRV", "BST", "BGW", "BHR", "BJB", "BLV", "BRC", "BSD", "BCT", "BMS"]

#Financial Categories (Actuals)
FinCatList = ["Sick Time", "Overtime", "Purchased Hours", "- Added Care", "Purchased Hours total", "Repair", "Maintenance", "Other (RnM)", "6500:Care Supplies", "6600:Clinical supplies", "Nursing Supplies - Added Care Expense", "SUPPLIES", "Added-Care", "6700:Repair & Maintenance", "SUM"]

#Financial Categories (Budget)
FinCatListBudget = ["Sick Time","Overtime","Purchased Hours","Repair","Maintenance","Other (RnM)","6500:Care Supplies","6600:Clinical supplies","SUPPLIES","6700:Repair & Maintenance","SUM"]

#Variance Categories
VarCatList = ["Sick Time", "Overtime", "Purchased Hours", "Repair", "Maintenance","Other (RnM)", "6500:Care Supplies", "6600:Clinical supplies", "SUPPLIES"]

#defined month sequence for improvement colours
MonthSequence = ["Jan 2025", "Feb 2025", "Mar 2025", "Apr 2025", "May 2025", "Jun 2025", "Jul 2025", "Aug 2025", "Sep 2025", "Oct 2025", "Nov 2025", "Dec 2025", "Jan 2026", "Feb 2026", "Mar 2026", "Apr 2026", "May 2026", "Jun 2026", "Jul 2026", "Aug 2026", "Sep 2026", "Oct 2026", "Nov 2026", "Dec 2026","Jan 2027", "Feb 2027", "Mar 2027", "Apr 2027", "May 2027", "Jun 2027", "Jul 2027", "Aug 2027", "Sep 2027", "Oct 2027", "Nov 2027", "Dec 2027","Jan 2028", "Feb 2028", "Mar 2028", "Apr 2028", "May 2028", "Jun 2028", "Jul 2028", "Aug 2028", "Sep 2028", "Oct 2028", "Nov 2028", "Dec 2028","Jan 2029", "Feb 2029", "Mar 2029", "Apr 2029", "May 2029", "Jun 2029", "Jul 2029", "Aug 2029", "Sep 2029", "Oct 2029", "Nov 2029", "Dec 2029", "Jan 2030", "Feb 2030", "Mar 2030", "Apr 2030", "May 2030", "Jun 2030", "Jul 2030", "Aug 2030", "Sep 2030", "Oct 2030", "Nov 2030", "Dec 2030","Jan 2031", "Feb 2031", "Mar 2031", "Apr 2031", "May 2031", "Jun 2031", "Jul 2031", "Aug 2031", "Sep 2031", "Oct 2031", "Nov 2031", "Dec 2031","Jan 2032", "Feb 2032", "Mar 2032", "Apr 2032", "May 2032", "Jun 2032", "Jul 2032", "Aug 2032", "Sep 2032", "Oct 2032", "Nov 2032", "Dec 2032"]

#defined site sequence for final columns order
SiteSequence = ['CL','GL','ACILDS','MN','SN','OM','VV','HC','SW','ST','SP','SG','WH','BCM','CPL','MOM','MIM','SR','BBC','BJP','BMT','BMC','BMR','BRV','BST','BGW','BHR','BJB','BLV','BRC','BSD','BCT','BMS','MAC','IAM','IAR','IEM','ILD','IPG','IPH','IRO','ITL','RBR','RGO','RSM','RWB','RWG','RWH','RWL','RWW','MGS','MBC','MQG','MRG','MVF']

#Flag for ACIL File
ACILFlag = {}
LedgerColumnsACIL = {}
MonthColumnsActualsACIL = {}
MonthColumnsBudgetsACIL = {}
MonthsListPossible = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar']


Now, let's load an Excel file into a pandas DataFrame. You can specify a particular worksheet by name or by index (0-based) using the `sheet_name` parameter in `pd.read_excel()`.

In [101]:
unique_site_names = set()
unique_month_names = set()

for filename in SourceFile: # Iterate through each file in SourceFile list
    full_file_path = SourceFilePath + filename

    # Read all sheets from the current Excel file
    # sheet_name=None returns a dictionary where keys are sheet names and values are DataFrames
    # header=None tells pandas that the file does not have a header row, so row 0 is data.
    current_file_worksheets = pd.read_excel(full_file_path, sheet_name=None, header=None)
    all_worksheets_df[filename] = current_file_worksheets # Store under the filename as per new definition

    # Populate SiteList and MonthList from sheets of the current file
    for sheet_name in current_file_worksheets.keys():
        #print("sheet_name:",sheet_name)
        if (re.fullmatch(r'^[A-Z]{2,3}$', sheet_name) and sheet_name != "YTD"): # Check for exactly two or three capital letters
            unique_site_names.add(sheet_name)
        elif (sheet_name == "ACILDS"): # Check for MMM 20XX Budget pattern
            unique_site_names.add(sheet_name)

        # Check for month budget worksheets and extract the month
        if re.fullmatch(r'^[A-Z][a-z]{2} 20[0-9]{2} Budget$', sheet_name): # Check for MMM 20XX Budget pattern
            unique_month_names.add(sheet_name[:8])

    # Convert sets to sorted lists for consistent output
    # Sort unique_site_names based on the order in SiteSequence
    SiteList[filename] = sorted(
        [site for site in unique_site_names if site in SiteSequence],
        key=lambda site: SiteSequence.index(site)
    )
    # Sort unique_month_names based on the order in MonthSequence
    MonthList[filename] = sorted(
        [month for month in unique_month_names if month in MonthSequence],
        key=lambda month: MonthSequence.index(month)
    )

#print(f"Successfully loaded all worksheets from file: {filename}.")
#print(f"Worksheets identified as Site Locations (two or three capital letters): {SiteList[filename]}")
#print(f"Months identified from 'MMM 20XX Budget' worksheets: {MonthList[filename]}")

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [102]:
#set Year string variable
YearString = ""
YearsOptions = ['2024','2025','2026','2027','2028','2029','2030','2031','2032']

#checking for ACIL file
for filename in SourceFile:

    # Check if filename exists in MonthList and if its corresponding list is not empty
    if filename in MonthList and MonthList[filename]:
        ACILFlag[filename] = False # It has specific budget months, not a generic ACIL file
        continue

    else:
        ACILFlag[filename] = True # Either filename not in MonthList or MonthList[filename] is empty, treat as generic ACIL

        # Initialize dictionaries for the current filename if they don't exist
        if filename not in LedgerColumnsACIL:
            LedgerColumnsACIL[filename] = {}
        if filename not in MonthColumnsActualsACIL:
            MonthColumnsActualsACIL[filename] = {}
        if filename not in MonthColumnsBudgetsACIL:
            MonthColumnsBudgetsACIL[filename] = {}

        #open each one identified site
        for site in SiteList[filename]:
            # Initialize site-specific dictionaries for month-based columns
            if site not in MonthColumnsActualsACIL[filename]:
                MonthColumnsActualsACIL[filename][site] = {}
            if site not in MonthColumnsBudgetsACIL[filename]:
                MonthColumnsBudgetsACIL[filename][site] = {}

            # Get the DataFrame for the current site
            ACIL_Site_df = all_worksheets_df[filename][site]

            #if Year string is blank then search current site worksheet for fiscal year
            if YearString == "":
              for c_idx in range(0, 4):
                for r_idx in range(0,13):
                  cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                  if isinstance(cell_value, str):
                      StartingFour = cell_value[0:4]
                      if StartingFour in YearsOptions:
                        YearString = StartingFour
                        break
                  if YearString != "":
                    break
                if YearString != "":
                  break

            # Find LedgerColumn
            current_ledger_col = FindLedgerColumn(ACIL_Site_df)
            LedgerColumnsACIL[filename][site] = current_ledger_col

            #Find Months and then Actual and Budget columns
            for monthcheck in MonthsListPossible:
                month_header_row, current_month_col = FindMonthActualsColumn(ACIL_Site_df, monthcheck) # Get both row and col
                if(current_month_col != -1): # If a column for the month was found
                    actuals_found = False
                    budget_found = False
                    actuals_value_found = False
                    budget_value_found = False
                    actuals_column_candidate = -1
                    budget_column_candidate = -1


                    # Look down the current_month_col for "Actuals" and "Budget" labels
                    # Start searching from the row where the month header was found, or the row after it.
                    for c_idx in range(current_month_col, current_month_col+2):

                      for r_idx in range(month_header_row, month_header_row+5):
                        cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                        if isinstance(cell_value, str):
                            upper_value = cell_value.strip().upper()
                            if upper_value == "ACTUALS" and actuals_found is False:
                                actuals_column_candidate = c_idx
                                actuals_found = True

                            if upper_value == "BUDGET" and budget_found is False:
                                budget_column_candidate = c_idx
                                budget_found = True

                        if(actuals_found and actuals_column_candidate == c_idx):
                            #check if current cell contains financial data
                            cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                            if isinstance(cell_value, (int, float)) and pd.notna(cell_value):
                                actuals_value_found = True

                        if(budget_found and budget_column_candidate == c_idx):
                            #check if current cell contains financial data
                            cell_value = ACIL_Site_df.iloc[r_idx, c_idx]
                            if isinstance(cell_value, (int, float)) and pd.notna(cell_value):
                                budget_value_found = True

                        if(actuals_found and budget_found and actuals_value_found and budget_value_found):
                            MonthColumnsActualsACIL[filename][site][monthcheck]= actuals_column_candidate
                            MonthColumnsBudgetsACIL[filename][site][monthcheck] = budget_column_candidate
                            AddThisMonth = monthcheck + " " + YearString
                            if AddThisMonth not in MonthList[filename]:
                              MonthList[filename].append(AddThisMonth)
                              if monthcheck == "Dec":
                                #add 1 to Yearstring
                                YearString = str(int(YearString) + 1)


To view the original formulas in the Excel cells, we'll use `openpyxl` directly. This will allow us to access the `formula` attribute of each cell.

In [103]:
# Loop through Months and create RevisedMonthList with a single (latest) Month that exists in both

for filename in SourceFile: # Assuming this cell is meant to process all files from SourceFile
    # Initialize RevisedMonthList for the current filename. It will store at most one month.
    RevisedMonthList[filename] = []

    # Iterate through MonthList in reverse order to find the latest valid month
    found_latest_valid_month = None
    # Check if MonthList[filename] exists and is not empty before iterating
    if filename in MonthList and MonthList[filename]:
        for current_month in reversed(MonthList[filename]):
            BudgetMonth = current_month[:3]

            MonthActualsNotFound = False
            # Check if SiteList[filename] exists and is not empty before iterating
            if filename in SiteList and SiteList[filename]:
                for site in SiteList[filename]:
                    current_site_df = all_worksheets_df[filename][site]
                    # Find LedgerColumn and MonthColumn for the current site using the current BudgetMonth
                    if FindMonthActualsColumn(current_site_df, BudgetMonth)[1] == -1:
                        MonthActualsNotFound = True
                        break # Break from site loop if actuals not found for this site
            else:
                MonthActualsNotFound = True # No sites to check, so consider actuals not found for all sites

            if not MonthActualsNotFound:
                # If actuals are found for all sites, this is the latest valid month
                found_latest_valid_month = current_month
                break # Break from month loop as we found the latest single month

    if found_latest_valid_month:
        RevisedMonthList[filename] = [found_latest_valid_month]
    # If no valid month is found, RevisedMonthList[filename] remains an empty list, as initialized above.

In [104]:
print(RevisedMonthList)

{'ACIL-source.xlsx': ['Apr 2026']}


In [105]:
## This will store all row data for all months before creating the final DataFrame
financial_summary_df = {}
# Loop through each loaded file
for filename in SourceFile:

  # Initialize all_actuals_summary_rows as a list for each file to collect its data
  all_actuals_summary_rows = []
  for current_month in RevisedMonthList[filename]:
      # Update BudgetMonth for the current iteration (this variable is used in other functions)
      BudgetMonth = current_month[:3]

    # Iterate through each financial category
      for category in FinCatList:
          # Initialize a dictionary for the current row, including 'Month' and 'Actuals Category'
          row_data = {'File':filename, 'Month': current_month, 'Category': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Get the DataFrame for the current site
              current_site_df = all_worksheets_df[filename][site]

              # Find LedgerColumn and MonthColumn for the current site using the current BudgetMonth
              current_ledger_col = FindLedgerColumn(current_site_df)
              if ACILFlag[filename] is True:
                current_month_col = MonthColumnsActualsACIL[filename][site][BudgetMonth]
              else:
                current_month_col = FindMonthActualsColumn(current_site_df, BudgetMonth)[1]

              # Calculate the sum for the current site and category
              calculated_sum = calculate_value(
                  DataWorkSheet=current_site_df,
                  LedgerColumnIndex=current_ledger_col,
                  MonthColumnIndex=current_month_col,
                  Category=category
              )
              # Add the calculated sum to the row data with the site name as the key
              row_data[site] = calculated_sum

          # Add the completed row data to the summary list for all months
          all_actuals_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_df[filename] = pd.DataFrame(all_actuals_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary DataFrame (Actuals for all months)-> Filename:",filename)
  print(financial_summary_df)
  print("\n")


Combined Financial Summary DataFrame (Actuals for all months)-> Filename: ACIL-source.xlsx
{'ACIL-source.xlsx':                 File     Month                               Category  \
0   ACIL-source.xlsx  Apr 2026                              Sick Time   
1   ACIL-source.xlsx  Apr 2026                               Overtime   
2   ACIL-source.xlsx  Apr 2026                        Purchased Hours   
3   ACIL-source.xlsx  Apr 2026                           - Added Care   
4   ACIL-source.xlsx  Apr 2026                  Purchased Hours total   
5   ACIL-source.xlsx  Apr 2026                                 Repair   
6   ACIL-source.xlsx  Apr 2026                            Maintenance   
7   ACIL-source.xlsx  Apr 2026                            Other (RnM)   
8   ACIL-source.xlsx  Apr 2026                     6500:Care Supplies   
9   ACIL-source.xlsx  Apr 2026                 6600:Clinical supplies   
10  ACIL-source.xlsx  Apr 2026  Nursing Supplies - Added Care Expense   
11  ACIL-sou

In [106]:
print(RevisedMonthList)

#

{'ACIL-source.xlsx': ['Apr 2026']}


In [107]:
# This will store all row data for all months before creating the final DataFrame
financial_summary_budget_df = {}

# Loop through each loaded file
for filename in SourceFile:
  all_budget_summary_rows = []
  # Loop through each month in RevisedMonthList to generate budget data for each
  for current_month in RevisedMonthList[filename]:
      # Update BudgetMonth for the current iteration
      BudgetMonth = current_month[:3]
      if ACILFlag[filename] is False:
        Budget_sheet_name = f"{current_month} Budget"
        worksheet_budget_data = all_worksheets_df[filename][Budget_sheet_name]
        MonthBudgets = BudgetMonth + " Budget"

      # Iterate through each financial category for budget
      for category in FinCatListBudget:
          # Initialize a dictionary for the current row, including 'Month' and 'Budget Category'
          row_data = {'File':filename, 'Month': current_month, 'Category': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Calculate the sum for the current site and category
              worksheet_data = all_worksheets_df[filename][site]
              col_ledger_account = FindLedgerColumn(worksheet_data)
              if ACILFlag[filename] is True:
                calculated_sum = calculate_budget_value_ACIL(
                    LedgerColumnIndex=col_ledger_account,
                    TargetColumnIndex=MonthColumnsBudgetsACIL[filename][site][BudgetMonth],
                    TargetCategory=category,
                    BudgetSheetData = worksheet_data
                )

              else:
                calculated_sum = calculate_budget_value(
                    SiteName=site,
                    LedgerColumnIndex=col_ledger_account,
                    TargetCategory=category,
                    BudgetSheetData = worksheet_budget_data
                )
                #print("Category:",category," :calculated_sum:",calculated_sum)
              # Add the calculated sum to the row data with the site name as the key
              row_data[site] = calculated_sum

          # Add the completed row data to the summary list for all months
          all_budget_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_budget_df[filename] = pd.DataFrame(all_budget_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary Budget DataFrame (for all months)-> Filename:",filename)
  print(financial_summary_budget_df[filename])


Combined Financial Summary Budget DataFrame (for all months)-> Filename: ACIL-source.xlsx
                File     Month                   Category          CL  \
0   ACIL-source.xlsx  Apr 2026                  Sick Time     3734.63   
1   ACIL-source.xlsx  Apr 2026                   Overtime     1155.57   
2   ACIL-source.xlsx  Apr 2026            Purchased Hours      261.12   
3   ACIL-source.xlsx  Apr 2026                     Repair     4768.50   
4   ACIL-source.xlsx  Apr 2026                Maintenance    14810.40   
5   ACIL-source.xlsx  Apr 2026                Other (RnM) -1000000.00   
6   ACIL-source.xlsx  Apr 2026         6500:Care Supplies      376.01   
7   ACIL-source.xlsx  Apr 2026     6600:Clinical supplies      413.61   
8   ACIL-source.xlsx  Apr 2026                   SUPPLIES      789.62   
9   ACIL-source.xlsx  Apr 2026  6700:Repair & Maintenance    19578.90   
10  ACIL-source.xlsx  Apr 2026                        SUM    19578.90   

           GL      ACILDS        

In [108]:
# This will store all row data for all months before creating the final DataFrame
financial_summary_variance_df = {}

# Loop through each loaded file
for filename in SourceFile:
  # This will store all row data for all months before creating the final DataFrame
  all_variance_summary_rows = []
  # Loop through each month in RevisedMonthList to generate variance data for each
  for current_month in RevisedMonthList[filename]:
      print("current_month:",current_month)
      # Iterate through each variance category
      for category in VarCatList:
          # Initialize a dictionary for the current row, including 'Month' and 'Variance Category'
          row_data = {'File':filename, 'Month': current_month, 'Variance': category}

          # Iterate through each site
          for site in SiteList[filename]:
              # Get the actuals value for the current site and category for the current month
              actuals_filter_category = "Purchased Hours total" if category == "Purchased Hours" else category
              actuals_selection = financial_summary_df[filename].loc[
                  (financial_summary_df[filename]['Month'] == current_month) &
                  (financial_summary_df[filename]['Category'] == actuals_filter_category), site
              ]
              # Safely get actuals_value, defaulting to 0.0 if not found
              actuals_value = actuals_selection.iloc[0] if not actuals_selection.empty else 0.0

              # Get the budget value for the current site and category for the current month
              budget_selection = financial_summary_budget_df[filename].loc[
                  (financial_summary_budget_df[filename]['Month'] == current_month) &
                  (financial_summary_budget_df[filename]['Category'] == category), site
              ]
              # Safely get budget_value, defaulting to 0.0 if not found
              budget_value = budget_selection.iloc[0] if not budget_selection.empty else 0.0

              # Calculate the variance (Budget - Actuals)
              variance = budget_value - actuals_value

              # Add the calculated variance to the row data with the site name as the key
              row_data[site] = variance

          # Add the completed row data to the summary list for all months
          all_variance_summary_rows.append(row_data)

  # Create the final DataFrame from all collected rows for all months
  financial_summary_variance_df[filename] = pd.DataFrame(all_variance_summary_rows)

  # Display the resulting combined DataFrame
  print("Combined Financial Summary Variance DataFrame (for all months)-> Filename:",filename)
  print(financial_summary_variance_df[filename])
  print("\n")



current_month: Apr 2026
Combined Financial Summary Variance DataFrame (for all months)-> Filename: ACIL-source.xlsx
               File     Month                Variance          CL          GL  \
0  ACIL-source.xlsx  Apr 2026               Sick Time    -1542.88    -1587.33   
1  ACIL-source.xlsx  Apr 2026                Overtime    -1713.19    -1659.27   
2  ACIL-source.xlsx  Apr 2026         Purchased Hours      208.62      -52.50   
3  ACIL-source.xlsx  Apr 2026                  Repair    -5534.49     -114.11   
4  ACIL-source.xlsx  Apr 2026             Maintenance     8529.32    -5132.57   
5  ACIL-source.xlsx  Apr 2026             Other (RnM) -1000000.00 -1000000.00   
6  ACIL-source.xlsx  Apr 2026      6500:Care Supplies       59.96    -6409.48   
7  ACIL-source.xlsx  Apr 2026  6600:Clinical supplies      413.61    -6364.29   
8  ACIL-source.xlsx  Apr 2026                SUPPLIES      473.57   -12773.77   

       ACILDS          MN          SN          OM          VV          HC

Now, let's create a new Excel file containing only the data values from the 'Payroll Budget' worksheet. We'll use the DataFrame `df` which already holds these values.

In [109]:
def FindFinCatColumn(DataWorkSheet):
  FinCat_col = 1000
  for r_idx, row in DataWorkSheet.iterrows():
      for c_idx, value in row.items():
          if isinstance(value, str) and value == "Actual":
              if(c_idx < FinCat_col):
                FinCat_col = c_idx
              continue
  return FinCat_col

In [110]:
def FindSiteColumnFromMonth(DataWorkSheet,Site):
  site_row = -1
  site_col = -1
  for r_idx, row in DataWorkSheet.iterrows():
      for c_idx, value in row.items():
          if isinstance(value, str) and value.strip().upper() == Site.strip().upper():
              site_row = r_idx
              site_col = c_idx
              break
      if site_row != -1:
          break
  return site_row, site_col

In [111]:
#define mapping from Region to Sites
RegionSiteMapping = {}
RegionSiteMapping['ACIL'] = ['CL','GL','ACILDS','MN','SN','OM','VV','HC']
RegionSiteMapping['Aster AB'] = ['SW','ST','SP','SG','WH']
RegionSiteMapping['Aster BC'] = ['BCM','CPL','MOM','MIM','SR']
RegionSiteMapping['Bloom AB'] = ['BBC','BJP','BMT','BMC','BMR','BRV','BST']
RegionSiteMapping['Bloom BC'] = ['BCT','BGW','BHR','BJB','BLV','BMS','BRC','BSD']
RegionSiteMapping['Iris'] = ['IAR','IAM','IEM','ILD','IPH','IPG','IRO','ITL']
RegionSiteMapping['Regency'] = ['RBR','RGO','RWL','RWB','RWW','RWG','RWH','RSM']

#define column listing for sites in lower summary
SiteColumnList = ['BBC','BJP','BMT','BMC','BMR','BRV','BST','BCT','BGW','BHR','BJB','BLV','BMS','BRC','BSD','','BCM','CPL','MOM','MIM','SR','SW','ST','SP','SG','WH','','IAR','IAM','IEM','ILD','IPH','IPG','IRO','ITL','','RBR','RGO','RWL','RWB','RWW','RWG','RWH','RSM']

In [112]:
#define mapping for BloomSites
BloomCategoryMapping = {}
BloomCategoryMapping['Actual'] = {}
BloomCategoryMapping['Budget'] = {}
BloomCategoryMapping['Variance'] = {}
BloomCategoryMapping['Compared to last'] = {}

BloomCategoryMapping['Actual']['Sick Time'] = 'Sick Time'
BloomCategoryMapping['Actual']['Overtime'] = 'Overtime'
BloomCategoryMapping['Actual']['Purchased Hours (- Added Care for AB only, or - podiatry for ON only)'] = 'Purchased Hours total'
BloomCategoryMapping['Actual']['6500 + 6600 SUPPLIES (- Added Care AB Only)'] = 'SUPPLIES'
BloomCategoryMapping['Actual']['Repairs (RnM funded ON)'] = 'Repair'
BloomCategoryMapping['Actual']['Maintenance (RnM unfunded ON)'] = 'Maintenance'
BloomCategoryMapping['Actual']['Other (R&M)'] = 'Other (RnM)'
BloomCategoryMapping['Actual']['CapEx'] = ''
BloomCategoryMapping['Actual'][' Supplies (not funded)'] = ''
BloomCategoryMapping['Actual']['BOTH R&M (for ACIL)'] = ''

BloomCategoryMapping['Budget']['Sick Time'] = 'Sick Time'
BloomCategoryMapping['Budget']['Overtime'] = 'Overtime'
BloomCategoryMapping['Budget']['Purchased Hours'] = 'Purchased Hours'
BloomCategoryMapping['Budget']['6500 + 6600 SUPPLIES'] = 'SUPPLIES'
BloomCategoryMapping['Budget']['Repairs (both R&M for ACIL) and (funded RnM ON)'] = 'Repair'
BloomCategoryMapping['Budget']['Maintenance (unfunded RnM ON)'] = 'Maintenance'
BloomCategoryMapping['Budget']['Other (R&M)'] = 'Other (RnM)'
BloomCategoryMapping['Budget']['Supplies (not funded)'] = ''

BloomCategoryMapping['Variance']['Sick Time'] = 'Sick Time'
BloomCategoryMapping['Variance']['Overtime'] = 'Overtime'
BloomCategoryMapping['Variance']['Purchased Hours'] = 'Purchased Hours'
BloomCategoryMapping['Variance']['6500 + 6600 SUPPLIES'] = 'SUPPLIES'
BloomCategoryMapping['Variance']['Repairs (both R&M for ACIL) (RnM funded ON)'] = 'Repair'
BloomCategoryMapping['Variance']['Maintenance (RnM unfunded ON)'] = 'Maintenance'
BloomCategoryMapping['Variance']['Other (R&M)'] = 'Other (RnM)'
BloomCategoryMapping['Variance']['Supplies (not funded)'] = ''




In [113]:
import numpy as np
# Open the NextMonthTemplateFile and load the single Worksheet: MonthYY
template_full_path = NextMonthTemplateFilePath + NextMonthTemplateFile
# Load the "MonthYY" sheet from the template file, assuming no explicit header.
month_template_df = pd.read_excel(template_full_path, sheet_name="MonthYY", header=None)

# Find Financial Categories column index from Worksheet: MonthYY
FinCatColumnIndex = FindFinCatColumn(month_template_df)

#find Indices for all sites in SiteList
site_column_indices = {}
for filename in SiteList:
  for site in SiteList[filename]:
    site_column_indices[site] = FindSiteColumnFromMonth(month_template_df,site)[1]

# Determine the month to fill into the template (the latest month identified from the source file).
target_month = RevisedMonthList[SourceFile[0]][0]

#print(target_month)

CategrorySectionList = ['Actual','Budget','Variance'] # Changed to 'Actual' to match template 'Actual'
BudgetSectionRowStart = 0
VarianceSectionRowStart = 0

# Walk through sites, financial categories for Actuals, and put values in corresponding cells
if target_month:
    for filename in SourceFile: # Assuming we fill the template based on data from the first source file
        if filename in financial_summary_df and filename in SiteList:
            monthly_actuals_df = financial_summary_df[filename][
                financial_summary_df[filename]['Month'] == target_month
            ]
            for category in FinCatList:
                #all FinCatList elements are only in the Actual CategorySection
                CategrorySection = 'Actual'
                # Find the row index for the current category in the template df
                category_row_index = -1

                for r_idx in range(month_template_df.shape[0]):
                    template_cell_value = month_template_df.iloc[r_idx, FinCatColumnIndex]
                    if(template_cell_value == "Budget"):
                      BudgetSectionRowStart = r_idx
                      break #all Actuals categories have been examined

                    # Now if the template_cell_value
                    # is a string that could potentially be a category mapping key.
                    if isinstance(template_cell_value, str):
                        # Ensure template_cell_value is a key in BloomCategoryMapping[CategrorySection]
                        # to prevent KeyError if it's some other non-category string.
                        if template_cell_value in BloomCategoryMapping[CategrorySection]:
                            # Only then attempt to access BloomCategoryMapping
                            # and check if its mapped value matches the 'category' we are looking for.
                            if isinstance(BloomCategoryMapping[CategrorySection][template_cell_value], str) and \
                                BloomCategoryMapping[CategrorySection][template_cell_value] == category:
                                category_row_index = r_idx
                                break # Found the category, exit the inner loop

                if category_row_index != -1:
                    for site in SiteList[filename]:
                        if site in site_column_indices:
                            site_col_idx = site_column_indices[site]
                            # Get the actuals value for the current category and site
                            actuals_value_selection = monthly_actuals_df.loc[
                                monthly_actuals_df['Category'] == category, site
                            ]
                            if not actuals_value_selection.empty:
                                actuals_value = actuals_value_selection.iloc[0]
                                # Put the value into the template DataFrame
                                month_template_df.iloc[category_row_index, site_col_idx] = actuals_value
                            else:
                                # If actuals value is not found, set to NaN in the template
                                month_template_df.iloc[category_row_index, site_col_idx] = np.nan

        if filename in financial_summary_budget_df and filename in SiteList:
            monthly_budget_df = financial_summary_budget_df[filename][
                financial_summary_budget_df[filename]['Month'] == target_month
            ]
            for category in FinCatListBudget:
                #all FinCatListBudget elements are only in the Budget CategorySection
                CategrorySection = 'Budget'
                # Find the row index for the current category in the template df
                category_row_index = -1

                for r_idx in range(month_template_df.shape[0]):
                    if(BudgetSectionRowStart > r_idx):
                      continue

                    template_cell_value = month_template_df.iloc[r_idx, FinCatColumnIndex]
                    if(template_cell_value == "Variance"):
                      VarianceSectionRowStart = r_idx
                      break #all Budget categories have been examined
                    # Now if the template_cell_value
                    # is a string that could potentially be a category mapping key.
                    if isinstance(template_cell_value, str):
                        # Ensure template_cell_value is a key in BloomCategoryMapping[CategrorySection]
                        # to prevent KeyError if it's some other non-category string.
                        if template_cell_value in BloomCategoryMapping[CategrorySection]:
                            # Only then attempt to access BloomCategoryMapping
                            # and check if its mapped value matches the 'category' we are looking for.
                            if isinstance(BloomCategoryMapping[CategrorySection][template_cell_value], str) and \
                                BloomCategoryMapping[CategrorySection][template_cell_value] == category:
                                category_row_index = r_idx
                                break # Found the category, exit the inner loop

                if category_row_index != -1:
                    for site in SiteList[filename]:
                        if site in site_column_indices:
                            site_col_idx = site_column_indices[site]
                            # Get the budget value for the current category and site
                            budget_value_selection = monthly_budget_df.loc[
                                monthly_budget_df['Category'] == category, site
                            ]
                            if not budget_value_selection.empty:
                                budget_value = budget_value_selection.iloc[0]
                                # Put the value into the template DataFrame
                                month_template_df.iloc[category_row_index, site_col_idx] = budget_value
                            else:
                                # If budget value is not found, set to NaN in the template
                                month_template_df.iloc[category_row_index, site_col_idx] = np.nan


        if filename in financial_summary_variance_df and filename in SiteList:
            monthly_Variance_df = financial_summary_variance_df[filename][
                financial_summary_variance_df[filename]['Month'] == target_month
            ]
            for category in VarCatList:
                #all VarCatList elements are only in the Variance Category Section
                CategrorySection = 'Variance'
                # Find the row index for the current category in the template df
                category_row_index = -1

                for r_idx in range(month_template_df.shape[0]):
                    if(VarianceSectionRowStart > r_idx):
                      continue
                    template_cell_value = month_template_df.iloc[r_idx, FinCatColumnIndex]
                    if(template_cell_value == "Compared to last"):
                      break #all Variance categories have been examined
                    # Now if the template_cell_value
                    # is a string that could potentially be a category mapping key.

                    #print("6template_cell_value:",template_cell_value)
                    if isinstance(template_cell_value, str):
                        # Ensure template_cell_value is a key in BloomCategoryMapping[CategrorySection]
                        # to prevent KeyError if it's some other non-category string.
                        if template_cell_value in BloomCategoryMapping[CategrorySection]:
                            # Only then attempt to access BloomCategoryMapping
                            # and check if its mapped value matches the 'category' we are looking for.
                            if isinstance(BloomCategoryMapping[CategrorySection][template_cell_value], str) and \
                                BloomCategoryMapping[CategrorySection][template_cell_value] == category:
                                category_row_index = r_idx
                                break # Found the category, exit the inner loop

                #print("category_row_index:",category_row_index)
                if category_row_index != -1:
                    for site in SiteList[filename]:
                        if site in site_column_indices:
                            site_col_idx = site_column_indices[site]
                            # Get the variance value for the current category and site
                            variance_value_selection = monthly_Variance_df.loc[
                                monthly_Variance_df['Variance'] == category, site
                            ]
                            if not variance_value_selection.empty:
                                variance_value = variance_value_selection.iloc[0]
                                # Put the value into the template DataFrame
                                month_template_df.iloc[category_row_index, site_col_idx] = variance_value
                            else:
                                # If variance value is not found, set to NaN in the template
                                month_template_df.iloc[category_row_index, site_col_idx] = np.nan

In [114]:
#write the MonthYY worksheet into new DestinationFile
new_excel_file_path = DestinationFilePath + DestinationFile
month_template_df.to_excel(new_excel_file_path, sheet_name=RevisedMonthList[SourceFile[0]][0], index=False, header=False)


In [115]:
STOP
# write the Summary File

# Create a Pandas Excel writer using XlsxWriter as the engine.
with pd.ExcelWriter(new_excel_file_path, engine='xlsxwriter') as writer:
    # Get the workbook object to add formats
    workbook = writer.book

    # Combine all months from all files, deduplicate, and sort them
    all_months = []
    for filename in SourceFile:
        all_months.extend(RevisedMonthList[filename])
    unique_sorted_months = sorted(list(set(all_months)), key=lambda x: MonthSequence.index(x))

    # Gradient formats for actuals increases (worse, red shades)
    actuals_increase_formats = [] # This list will hold formats for RED gradient (worse)
    start_color_red = '#FFFFFF' # White
    end_color_red = '#FF0000' # Bright Red
    for i in range(100):
        factor = i / 99.0 # Factor from 0.0 to 1.0
        bg_color = interpolate_color(start_color_red, end_color_red, factor)
        font_color = get_contrasting_font_color(bg_color)
        actuals_increase_formats.append(workbook.add_format({'bg_color': bg_color, 'font_color': font_color, 'num_format': '0.00'}))

    # Gradient formats for actuals decreases (better, green shades)
    actuals_decrease_formats = [] # This list will hold formats for GREEN gradient (better)
    start_color_green = '#FFFFFF' # White
    end_color_green = '#008000' # Dark Green
    for i in range(100):
        factor = i / 99.0
        bg_color = interpolate_color(start_color_green, end_color_green, factor)
        font_color = get_contrasting_font_color(bg_color)
        actuals_decrease_formats.append(workbook.add_format({'bg_color': bg_color, 'font_color': font_color, 'num_format': '0.00'}))

    # Variance formats (simple red/green for now, will be replaced by gradients)
    red_format_variance_negative = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'num_format': '0.00'})
    green_format_variance_positive = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'num_format': '0.00'})

    default_number_format = workbook.add_format({'num_format': '0.00'})
    header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    category_format = workbook.add_format({'bold': True})

    # Cache for previous month's actuals DataFrame to enable comparison (reset for each workbook if needed)
    previous_month_actuals_df_cache = None

    # Now iterate through the unique, sorted list of months to create worksheets
    for idx, Worksheetmonth in enumerate(unique_sorted_months):
        # Create a worksheet for the current month
        worksheet = workbook.add_worksheet(Worksheetmonth)

        # Loop through each loaded file to populate data for the current month
        for filename in SourceFile:

            # Check if Worksheetmonth exists in RevisedMonthList[filename], continue otherwise
            if Worksheetmonth not in RevisedMonthList[filename]:
                continue

            # Filter data for the current month and drop 'File' and 'Month' columns
            # Keep 'Category' for actuals and budget, 'Variance' for variance
            monthly_actuals_df_filtered = financial_summary_df[filename][financial_summary_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month'])
            monthly_budget_df_filtered = financial_summary_budget_df[filename][financial_summary_budget_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month'])
            monthly_variance_df_filtered = financial_summary_variance_df[filename][financial_summary_variance_df[filename]['Month'] == Worksheetmonth].drop(columns=['File', 'Month']) # 'Variance' is the category-like column

            # --- Write monthly_actuals_df ---
            startrow_actuals = 0 # Starting row for actuals
            # Write header for actuals
            for col_num, value in enumerate(monthly_actuals_df_filtered.columns.values):
                worksheet.write(startrow_actuals, col_num, value, header_format)

            # Write data rows for actuals with conditional formatting
            for row_num_df, (index, row_data) in enumerate(monthly_actuals_df_filtered.iterrows()):
                excel_row = startrow_actuals + row_num_df + 1 # Excel row, after header
                worksheet.write(excel_row, 0, row_data['Category'], category_format) # Write 'Category' column

                # Compare and write site columns (starting from the second column of the filtered df)
                # The first column is 'Category', so `columns[1:]` gives site names.
                for col_num_df, col_name in enumerate(monthly_actuals_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1 # Excel column, after Category column

                    current_value = row_data[col_name]
                    format_to_apply_actuals = default_number_format # Default format

                    if idx > 0 and previous_month_actuals_df_cache is not None:
                        # Ensure previous_month_actuals_df_cache is also filtered similarly
                        prev_row_selection = previous_month_actuals_df_cache[previous_month_actuals_df_cache['Category'] == row_data['Category']]
                        previous_value = prev_row_selection.iloc[0].get(col_name) if not prev_row_selection.empty else None

                        if previous_value is not None and pd.notna(previous_value) and pd.notna(current_value):
                            if current_value > previous_value: # Actuals INCREASE (worse, red gradient)
                                percent_change = 0.0
                                if previous_value != 0:
                                    percent_change = ((current_value - previous_value) / previous_value) * 100
                                else:
                                    # If previous_value was 0 and current_value is > 0, it's an infinite increase (worse)
                                    percent_change = 200.0 # Map to darkest red for infinite increase from zero

                                # Map 0-200% increase to 0-99 index for the red gradient
                                # Cap percentage at 200% to fit the 100-step gradient nicely.
                                gradient_index = int(min(99, max(0, percent_change / 2)))
                                format_to_apply_actuals = actuals_increase_formats[gradient_index] # This is the RED gradient

                            elif current_value < previous_value: # Actuals DECREASE (better, green gradient)
                                percent_change = 0.0
                                if previous_value != 0:
                                    percent_change = ((previous_value - current_value) / abs(previous_value)) * 100
                                else:
                                    # If previous_value was 0 and current_value is < 0, it's an infinite decrease (better if it's an expense)
                                    percent_change = 100.0 # Map to darkest green for infinite decrease from zero

                                # Map 0-100% decrease to 0-99 index for the green gradient
                                # Cap percentage at 100% to fit the 100-step gradient nicely.
                                gradient_index = int(min(99, max(0, percent_change)))
                                format_to_apply_actuals = actuals_decrease_formats[gradient_index] # This is the GREEN gradient
                            else: # current_value == previous_value
                                format_to_apply_actuals = default_number_format

                    # Write the number with the chosen format, handling NaN values
                    if pd.isna(current_value):
                        worksheet.write_blank(excel_row, excel_col, None, format_to_apply_actuals)
                    else:
                        worksheet.write_number(excel_row, excel_col, current_value, format_to_apply_actuals)

            # Cache current actuals for the next iteration (store the filtered version)
            # This cache should be within the unique_sorted_months loop, but its use seems to imply a single previous month for comparison,
            # which means this logic needs careful consideration if 'previous_month_actuals_df_cache' is meant to be across files.
            # For now, it's reset per unique month, but if cross-file previous month comparison is intended, this needs adjustment.
            # For now, let's assume it should cache per month, for comparison to the 'previous' *unique* month.
            previous_month_actuals_df_cache = monthly_actuals_df_filtered.copy() # This cache will store the last df for the current month across all files. Needs to be more robust.

            # --- Write monthly_budget_df ---
            startrow_budget_monthly = startrow_actuals + len(monthly_actuals_df_filtered) + 2
            # Write header for budget
            for col_num, value in enumerate(monthly_budget_df_filtered.columns.values):
                worksheet.write(startrow_budget_monthly, col_num, value, header_format)
            # Write data rows for budget
            for row_num_df, (index, row_data) in enumerate(monthly_budget_df_filtered.iterrows()):
                excel_row = startrow_budget_monthly + row_num_df + 1
                worksheet.write(excel_row, 0, row_data['Category'], category_format)
                for col_num_df, col_name in enumerate(monthly_budget_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1
                    budget_value = row_data[col_name]
                    if pd.isna(budget_value):
                        worksheet.write_blank(excel_row, excel_col, None, default_number_format)
                    else:
                        worksheet.write_number(excel_row, excel_col, budget_value, default_number_format)


            # --- Write monthly_variance_df --- (Apply gradient formatting)
            startrow_variance_monthly = startrow_budget_monthly + len(monthly_budget_df_filtered) + 2
            # Write header for variance
            for col_num, value in enumerate(monthly_variance_df_filtered.columns.values):
                worksheet.write(startrow_variance_monthly, col_num, value, header_format)

            # Write data rows for variance with gradient conditional formatting
            for row_num_df, (index, row_data) in enumerate(monthly_variance_df_filtered.iterrows()):
                excel_row = startrow_variance_monthly + row_num_df + 1
                category_name = row_data['Variance'] # For variance, this column holds the variance category
                worksheet.write(excel_row, 0, category_name, category_format)

                for col_num_df, col_name in enumerate(monthly_variance_df_filtered.columns[1:]):
                    excel_col = col_num_df + 1
                    variance_value = row_data[col_name]

                    format_to_apply_variance = default_number_format # Default format

                    if pd.notna(variance_value): # Only apply conditional formatting if value is not NaN
                        if variance_value > 0: # Positive variance (better, green gradient)
                            # Map 0 to 30000 to 0-99 index for the green gradient
                            factor = min(1.0, max(0.0, variance_value / 30000.0))
                            gradient_index = int(factor * 99)
                            format_to_apply_variance = actuals_decrease_formats[gradient_index]

                        elif variance_value < 0: # Negative variance (worse, red gradient)
                            # Map -30000 to 0 to 0-99 index for the red gradient
                            factor = min(1.0, max(0.0, abs(variance_value) / 30000.0))
                            gradient_index = int(factor * 99)
                            format_to_apply_variance = actuals_increase_formats[gradient_index]

                    # Write the number with the chosen format, handling NaN values
                    if pd.isna(variance_value):
                        worksheet.write_blank(excel_row, excel_col, None, format_to_apply_variance)
                    else:
                        worksheet.write_number(excel_row, excel_col, variance_value, format_to_apply_variance)

    print(f"Successfully created new Excel file: {DestinationFile} in {DestinationFilePath}")
    #print("This file contains Actuals, Budget, and Variance data, with each month on its own sheet.")

NameError: name 'STOP' is not defined